In [3]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
from tqdm import tqdm
from scipy.stats import norm
from scipy.optimize import minimize
from copy import deepcopy
from sklearn.linear_model import LinearRegression
import torch
from scipy.stats import ttest_ind
import torch.nn as nn

In [4]:
dim_feature = 4
K = 5
Z = norm.ppf(0.975)
N = 100
mean_feature = 1/4
T = 1000
S = 2
number_per_split = int((K+N)/S)
noise_list = [1,2,3,4,5]

In [7]:
cost = np.zeros((len(noise_list),2,T))

N = K + N
for id,std in enumerate(noise_list):
    for t in tqdm(range(T)):
    # constant linear true function
        constant_linear = np.random.uniform(-0.3,0.5,dim_feature)

        # linear true function
        cofficent_linear = np.random.uniform(-0.3,0.5,(K,dim_feature))

        true_tao = np.sum(cofficent_linear[:,:]*0.5, axis=1)
        optimal_cost = np.sum(true_tao[np.argwhere(true_tao>0)])
        #optimal_cost[t] = np.sum(true_ate[np.argwhere(true_ate>0)])

        #generate data
        #Hist_feature = np.random.normal(mean_feature,3,(K+N,dim_feature))
        Hist_feature = np.random.uniform(0,1,(N,dim_feature))
        Hist_treatment = np.random.binomial(1,0.5,(N,K+1))
        Hist_treatment[:,0] = 1
        Hist_label = np.dot(Hist_feature,constant_linear.T) + np.sum(np.dot(Hist_feature,cofficent_linear.T)*Hist_treatment[:,1:],axis = 1) + np.random.normal(0,std,N)
        #Hist_feature[:,0] = 1

        



        #----DiM Method -----
        X = np.zeros((K,N))
        Y = np.zeros((K,N))
        for k in range(K):
            X[k,:] = Hist_treatment[:,k+1]
            Y[k,:] = Hist_label
        
        #DM
        tao_hat = np.zeros(K)
        variance = np.zeros(K)
        #variance1 = np.zeros(K)
        p_value_list = np.zeros(K)
        for k in range(K):
            group_1 = Y[k,X[k,:]==1]
            group_0 = Y[k,X[k,:]==0]
            t_stat, p_value = ttest_ind(group_1, group_0, equal_var = False)  
            diff_mean = group_1.mean() - group_0.mean()
            tao_hat[k] = diff_mean
            p_value_list[k] = p_value
            variance[k] = N*(group_1.var(ddof=1) / len(group_1) + group_0.var(ddof=1) / len(group_0))
        

        tao_0 = np.mean(tao_hat)
        numerator = np.mean(variance)
        denumerator = np.mean((tao_hat - tao_0)**2) - numerator/N

        # bayesian_tao = np.zeros(K)
        # bayesian_beta = np.zeros(K)

        # decision3 = []
  
        # for k in range(K):
        #     if denumerator <= 0:
        #         theta = 1
        #         posteri_mean = tao_hat[k]*theta + (1 - theta)*tao_0
        #         posteri_var = 1/(N/variance[k])
        #         dist = norm(loc=posteri_mean, scale=np.sqrt(posteri_var))
        #         prob = dist.sf(0)  # survival function: P(X > x)
        #         if prob > 1 - 0.025:
        #             decision3.append(k)
        #     else:
        #         bayesian_beta[k] = max(variance[k]/denumerator,0)
        #         theta = N/(N+bayesian_beta[k])
        #         posteri_mean = tao_hat[k]*theta + (1 - theta)*tao_0
        #         posteri_var = 1/(1/denumerator+ N/variance[k])
        #         dist = norm(loc=posteri_mean, scale=np.sqrt(posteri_var))

        #         prob = dist.sf(0)  # survival function: P(X > x)
        #         if prob > 1 - 0.025:
        #             decision3.append(k)


        
      
        beta = numerator/denumerator + Z*np.sqrt(N*numerator)/tao_0
        beta = max(0,beta)
        theta = N/(N+beta)
        #shunken_saa
        tao_shunken_hat = np.zeros(K)
        p_value_list_shrunken = np.zeros(K)
        Y_shunken = deepcopy(Y)
        Y_shunken = theta*Y_shunken 
        for k in range(K):
            group_1 = Y_shunken[k,X[k,:]==1] + (1-theta)*tao_0
            group_0 = Y_shunken[k,X[k,:]==0]
            t_stat, p_value = ttest_ind(group_1, group_0, equal_var = False)  
            diff_mean1 = group_1.mean() - group_0.mean()
            tao_shunken_hat[k] = diff_mean1
            p_value_list_shrunken[k] = p_value
        
    
        decision1 = np.intersect1d(np.argwhere(p_value_list<0.05), np.argwhere(tao_hat>0))
        decision2 = np.intersect1d(np.argwhere(p_value_list_shrunken<0.05), np.argwhere(tao_shunken_hat>0))

        if optimal_cost == 0:
            if len(decision1) == 0:
                cost[id,0,t] = 1
            else:
                cost[id,0,t] = 0
        else:
            cost[id,0,t] = np.sum(true_tao[decision1])/optimal_cost
        


    
        if optimal_cost == 0:
            if len(decision2) == 0:
                cost[id,1,t] = 1
            else:
                cost[id,1,t] = 0
        else:
            cost[id,1,t] = np.sum(true_tao[decision2])/optimal_cost
        

100%|██████████| 1000/1000 [00:03<00:00, 271.31it/s]


In [8]:
a = np.mean(cost,axis=2)
y1 = a[:,0]
y2 = a[:,1]
y1,y2

(array([0.436466  , 0.17177128, 0.09533569, 0.0768417 , 0.06066324]),
 array([0.71309882, 0.46278611, 0.36096898, 0.33959455, 0.31579977]))